**Nineteen.** And the eight opponent screen we had been ranking candidates with turned out to be four.

This notebook is about choosing the set of opponents you evaluate against. It is the piece that was missing from the two before it, and it exists because measuring our own pool returned a number that revised our own results downward.

### Where this sits

1. [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) measured that the field is homogeneous: most seats share a schedule for most of the game.
2. [Measure your agent](https://www.kaggle.com/code/destbreso/kaggriculture-measure-your-agent) is the harness that scores a candidate against a pool, and it does not justify how the pool was chosen. This notebook is that missing justification.
3. [Wins, not money](https://www.kaggle.com/code/destbreso/wins-not-money) uses an opponent set built by the method argued for here.
4. **This notebook** is the consequence of putting those two together. If the field is homogeneous, then a pool sampled from the field is homogeneous, and a win rate over n opponents is not worth n opponents.

### What this notebook does not claim

It does not claim an optimal pool, and it does not claim that the correction below is the only one our numbers need. It reports a property we failed to check for weeks, the arithmetic that makes it matter, and the three objectives that make it a design problem rather than a fix.

## 1. The problem

You want to know how a candidate performs against **the field**. You cannot play the field, so you play a sample of it and treat the sample as a stand in.

Three things break that substitution, and only the first is obvious.

- **The sample is not the field.** Standard sampling error, and everyone accounts for it.
- **The sample is not what you will meet.** Matchmaking pairs on rating, so the opponents you face are conditioned on your own position. A pool drawn from the whole ladder answers a question nobody asked.
- **The sample is clustered.** Many opponents play near identical routes. This one is invisible: the pool has forty names in it, and forty names look like forty opponents.

In [ ]:
# Chart style, matched to the companion notebooks: recessive axes, a light
# horizontal grid, nothing decorative. Colour is assigned by the job it does.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e7e7e4'
plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 130,
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white'})

def tidy(ax, ylab=None, title=None):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.xaxis.grid(False)
    if ylab: ax.set_ylabel(ylab)
    if title: ax.set_title(title, loc='left', pad=10, fontweight='bold')
    return ax

In [ ]:
import json
from itertools import combinations

def canon(a):
    return json.dumps(a, sort_keys=True, separators=(',', ':'))

def agreement(x, y):
    """Fraction of turns on which two recorded opponents act identically."""
    n = min(len(x), len(y))
    return sum(canon(x[t]) == canon(y[t]) for t in range(n)) / n

def cluster(streams, threshold=0.90):
    """Single linkage: two opponents are one behaviour if they mostly agree."""
    parent = list(range(len(streams)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j in combinations(range(len(streams)), 2):
        if agreement(streams[i], streams[j]) >= threshold:
            a, b = find(i), find(j)
            if a != b:
                parent[a] = b
    groups = {}
    for i in range(len(streams)):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())

# A pool that LOOKS like twelve opponents.
shared = [{'a': t % 5} for t in range(200)]                       # one popular route
pool, names = [], []
for k in range(7):                                                # seven copies of it,
    v = list(shared); v[10 * k] = {'a': 99}                       # each with a tiny quirk
    pool.append(v); names.append(f'copy_{k}')
for k in range(5):                                                # five genuine originals
    pool.append([{'a': (t * (k + 2)) % 7} for t in range(200)])
    names.append(f'original_{k}')

groups = cluster(pool)
print(f'names in the pool        {len(pool)}')
print(f'distinct behaviours      {len(groups)}')
print(f'cluster sizes            {sorted((len(g) for g in groups), reverse=True)}')

## 2. What our real pool looks like

The same procedure on 40 real recorded opponents, compared turn by turn and clustered at 90 % agreement.

| the pool, measured | |
|---|---|
| names in the pool | **40** |
| distinct behaviours | **19** |
| largest cluster | **18** |
| second cluster | 5 |
| singletons | 17 |
| median pairwise agreement | 17.5 % |
| mean pairwise agreement | 40.9 % |

**Twenty three of the forty collapse into two behaviours.** Five separate team names are one route.

The gap between a median of 17.5 % and a mean of 40.9 % is the shape worth recognising. Most pairs really are different. A large minority are near duplicates, and they drag the mean while leaving the median alone. Either statistic quoted by itself hides the structure entirely.

In [ ]:
# Measured on the 40 real recorded opponents: all 780 pairwise agreements,
# and the cluster sizes they produce at a 90 % threshold.
HIST  = [388, 25, 22, 3, 9, 18, 15, 41, 159, 100]
SIZES = [18, 5] + [1] * 17

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.7))

ax = axes[0]
edges = np.arange(10) * 10
cols = [ORANGE if e >= 80 else BLUE for e in edges]
ax.bar(edges + 5, HIST, width=9, color=cols)
tidy(ax, 'pairs of opponents', 'Two populations, not one')
ax.set_xlabel('turn by turn agreement, %')
ax.annotate('near duplicates\n259 of 780 pairs', (85, 159), xytext=(-14, 22),
            textcoords='offset points', fontsize=8.5, color=ORANGE,
            fontweight='bold', ha='right')
ax.annotate('median 17.5 %\nmean 40.9 %', (12, 330), fontsize=8.5, color=MUTED)

ax = axes[1]
ax.bar(range(len(SIZES)), SIZES, color=[ORANGE, ORANGE] + [BLUE] * 17, width=0.8)
tidy(ax, 'opponents in the cluster', '40 names, 19 behaviours')
ax.set_xlabel('cluster')
ax.set_xticks(range(0, len(SIZES), 3))
ax.annotate('one route,\n18 names', (0, 18), xytext=(10, -4),
            textcoords='offset points', fontsize=8.5, color=ORANGE,
            fontweight='bold')
plt.tight_layout(); plt.show()

## 3. The arithmetic, and what it did to our own results

Clustered sampling has a standard correction. With average cluster size *m* and intra cluster correlation *rho*, the design effect is `1 + (m - 1) * rho`, and the effective sample size is `n / deff`. Confidence intervals scale by the square root of that.

The practical version is simpler: **count behaviours, not names.**

In [ ]:
from math import sqrt

# deff = 1 + (m - 1) * rho, with m the mean cluster size and rho the intra-cluster
# correlation. Setting rho = 1 treats members of a cluster as interchangeable, which
# makes n_eff exactly the number of clusters. That is the CONSERVATIVE end of the
# range and it is the assumption used below; at rho = 0.7 the inflation would be
# 1.33x rather than 1.45x.
def deff(n, clusters, rho=1.0):
    m = n / clusters
    return 1 + (m - 1) * rho

print(f"{'instrument':<22}{'names':>7}{'games':>8}{'behaviours':>12}{'CI inflation':>14}")
for label, names_n, behaviours in [('screen (8 opponents)', 8, 4),
                                   ('field test (40)', 40, 19)]:
    print(f'{label:<22}{names_n:>7}{2*names_n:>8}{2*behaviours:>12}'
          f'{sqrt(deff(names_n, behaviours)):>13.2f}x')

print()
print('So a 16 game screen result carries the weight of 8 games,')
print('and any interval quoted on it is too narrow by about 41 %.')
print()
print(f"at rho = 0.7 instead of 1.0 the field-test inflation is "
      f"{sqrt(deff(40, 19, 0.7)):.2f}x rather than {sqrt(deff(40, 19)):.2f}x")

### What survived the correction and what did not

We applied this to a decision we had just made, which is the only honest way to introduce a correction.

**Survived.** A challenger had been rejected on a paired sign test of p = 0.56 over 80 games. Cutting the effective sample from 80 to 38 only lowers power, so a result that was already not significant becomes more so. The decision was right and is now better supported.

**Survived untouched.** The head to head test varies *seeds* rather than opponents, 20 seeds against one fixed adversary, so pool redundancy does not apply to it at all.

**Did not survive unqualified.** Every candidate ordering our screen ever produced rests on four behaviours. The selections may still be correct. The evidence for them is roughly half of what we reported, and we reported it as though the games were independent.

## 4. Why a pool inherits this, structurally

A pool built from your own recorded matches is a sample of the ladder, and [the ladder is measurably homogeneous](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening): most seats share a schedule for most of the opening, and unrelated strong teams play large fractions of the same publicly posted route.

Sampling uniformly from a clustered population gives a sample whose effective size is set by the number of clusters, not the number of draws. So the redundancy is not an accident of which opponents we happened to draw. **Any pool sampled from this field will have it**, and the more the field converges the worse it gets.

There is a second inheritance, easy to miss because it points the other way. A pool of *your own* matches is conditioned on your own rating, because matchmaking pairs on rating. It is therefore not a sample of the field at all. It is a sample of the field **at your level**, which is the right frame for predicting your next game and the wrong one for ranking candidates that might belong at a different level.

## 5. Three objectives that pull against each other

Knowing the pool is redundant does not tell you what pool to build, because there are three things you want and they conflict.

| objective | what it asks for | what it costs |
|---|---|---|
| **coverage** | one opponent per distinct behaviour | over weights rare behaviours you will seldom meet |
| **representativeness** | opponents in proportion to how often you meet them | in a homogeneous field, that is mostly duplicates |
| **discrimination** | opponents that beat you | unrepresentative by construction, and the hardest to collect |

The tension is sharpest exactly where we are. A representative sample of a field you mostly beat contains almost no games that carry ranking information, and a pool of opponents who beat you tells you nothing about the field.

**Our pool was chosen for none of the three.** It is whoever matchmaking happened to hand us.

In [ ]:
import random
random.seed(7)

# Same budget, two ways of spending it: naive draw versus one per cluster.
clusters = {'popular': 18, 'common': 5, **{f'rare_{i}': 1 for i in range(17)}}
population = [name for name, k in clusters.items() for _ in range(k)]
BUDGET = 8

naive = random.sample(population, BUDGET)
stratified = list(dict.fromkeys(sorted(clusters, key=lambda c: -clusters[c])))[:BUDGET]

print(f'budget: {BUDGET} opponents, identical cost either way\n')
print(f'naive draw       {len(set(naive))} behaviours   {sorted(set(naive))}')
print(f'one per cluster  {len(set(stratified))} behaviours   {sorted(stratified)}')
print()
print('Stratifying is free. It changes which opponents you play, not how many.')

## 6. What we changed

1. **Report effective n, never nominal n.** Cluster the pool and print behaviours next to games. Saying nothing about redundancy reads exactly like a claim of independence.
2. **Deduplicate the screen.** Eight opponents drawn one per cluster strictly dominates eight drawn arbitrarily, at the same cost.
3. **Weight by how often we actually meet each behaviour**, rather than by how many names carry it.
4. **Keep a deliberately hard tier.** A screen you always win ranks nothing, which is what saturation was telling us all along.
5. **Re run the selection this correction undermines**, rather than assuming it survives.

### Limits

A 90 % agreement threshold is a choice, and single linkage clustering will chain. We have not tested how the clusters move under either. The correction here is a floor on how wrong the old numbers were, not a precise restatement of them. And every number in this notebook is still offline: it improves how we sample opponents, and it says nothing about whether our simulated pool resembles the live ladder at all.

---

*Written during the competition. The specific selection results this apparatus produced are held back until after it closes; the apparatus itself is described in full because that is the part which generalises.*

---

## Appendix: pitfalls

Kept separate from the argument. Each is invisible in the output it corrupts, which is why none of them is caught by reading the code.

| pitfall | why it is invisible | the check |
|---|---|---|
| **Reusing a dataset built for another question** | it is correct data, correctly collected | re-derive the properties the new use requires, not the old one |
| **Counting names instead of behaviours** | 40 rows look like 40 opponents | cluster first, report effective n |
| **An opponent set you always beat** | it saturates, and saturation reads as strength | keep a tier that beats you, or the screen ranks nothing |
| **Evaluating a candidate against a copy of itself** | the comparison returns a clean, meaningless tie | hash and cluster candidates against what you already field |
| **Quoting a binomial interval on clustered games** | the interval is arithmetically correct | scale it by the square root of the design effect |

The shape they share: the quantity computed is right, and it answers a question one step from the one being asked. No code review catches that, because there is nothing wrong with the code.

---

**Engine version.** The answer here is **build-independent**, which is worth saying explicitly given that the game was rebalanced on 2026-08-15. Everything above counts *behaviours*, by clustering opponents on how their action streams agree turn by turn. That is a property of what people played, not of what the market paid them for it, so the change to the carrot, tomato and egg price curves cannot move it. What a rebalance can do is change the field itself, since people resubmit; the count of nineteen is a measurement of one pool at one moment, and the section on how a pool inherits this is the part that generalises. [What 1.32.7 changed, and what it did not](https://www.kaggle.com/code/destbreso/what-1-32-7-changed-and-what-it-did-not) separates what moved from what did not.